# Cache playground

Load the LLM's elicited variables + estimates for **one question** from an
elicitation cache, fit the flow solver on them with either the plain
**entropy** objective (`domain_prior="uniform"`) or the **gaussian-KL**
domain prior (`domain_prior="gaussian"`, see `gaussian_kl_objective.md`),
and look at what comes out.

**How to use:** edit the EDIT blocks (cache, question, solver mode), then
Run All. Run from the repo root (same as `playground.ipynb`).
Caches are $-expensive and immutable — this notebook only reads them.

In [ ]:
import os, sys, json
os.environ.setdefault("JAX_PLATFORMS", "cpu")   # remove on a GPU runtime
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("metaculus").resolve()))
from run_flow_solver import deserialize, direct_llm_estimate
from run_elicitation import load_dataset, entry_key, entry_question, TARGET_NAME
from calibrated_response.maxent_sampler import DistributionBuilder, plot_pairwise
from calibrated_response.generation.protocol import collapse_repeats

# ---- A) EDIT: which cache --------------------------------------------------
PROTOCOL = "baseline"     # baseline / v1 / v1x2 / v1_fermi / v1_spread / full
CACHE = Path("metaculus/caches") / PROTOCOL / f"llm_cache_{PROTOCOL}.json"

cache = json.loads(CACHE.read_text(encoding="utf-8"))
_, entries = load_dataset("metaculus/data/full_dataset.json")
by_key = {entry_key(e): e for e in entries}

for i, k in enumerate(cache):
    q = entry_question(by_key[k]) if k in by_key else "(not in dataset)"
    print(f"[{i:>3}] {k:<28} {q[:70]}")

In [ ]:
# ---- A) EDIT: which question ------------------------------------------------
PICK = 0                  # index from the list above, or a full 'id@round' key

key = PICK if isinstance(PICK, str) else list(cache)[PICK]
entry = by_key.get(key, {})
variables, estimates = deserialize(cache[key])

print(entry_question(entry) if entry else key)
print(f"resolved_to: {entry.get('resolved_to')}   "
      f"direct LLM: {direct_llm_estimate(estimates)}\n")
for v in variables:
    rng = (f" in [{v.lower_bound}, {v.upper_bound}] {getattr(v, 'unit', '') or ''}"
           if hasattr(v, "lower_bound") else " (binary)")
    print(f"  {v.name}{rng} — {(v.description or '')[:60]}")
print()
for est in estimates:
    print("  ", est.to_query_estimate())

In [ ]:
# ---- B) EDIT + FIT -----------------------------------------------------------
DOMAIN_PRIOR = "gaussian"   # "uniform" = plain maxent; "gaussian" = KL to N(mid, span/(2k))
PRIOR_BOUND_SDS = 2.0       # k: elicited bounds sit at ±k sd of the default belief
ENTROPY_REG = 1.0           # weight of the entropy / KL term
COLLAPSE = True             # fold repeated estimates (the solver default)

ests = collapse_repeats(estimates, prob_logit_sd=0.3) if COLLAPSE else estimates
builder = DistributionBuilder(variables, ests, prob_penalty="logit",
                              domain_prior=DOMAIN_PRIOR,
                              prior_bound_sds=PRIOR_BOUND_SDS)
builder.fit(steps=1500, n_samples=2048, entropy_reg=ENTROPY_REG, seed=0)

for w in builder.skipped + builder.warnings:
    print("!", w)

print(f"\n{'estimate':<52} {'target':>8} {'fitted':>8} {'p_cond':>7}")
for r in builder.constraint_report(n_samples=50_000):
    flag = " <-- check" if abs(r["error_rel"]) > 0.05 else ""
    pc = f"{r['p_cond']:.3f}" if r["p_cond"] is not None else "     -"
    print(f"{r['estimate'][:52]:<52} {r['target']:>8.3f} "
          f"{r['fitted']:>8.3f} {pc:>7}{flag}")

In [ ]:
# ---- C) READOUT ---------------------------------------------------------------
p = builder.marginal(TARGET_NAME).probability
kl = builder.kl_to_ref()
print(f"P(target)  = {p:.3f}   ({DOMAIN_PRIOR} prior)")
print(f"direct LLM = {direct_llm_estimate(estimates)}   "
      f"resolved_to: {entry.get('resolved_to')}")
print(f"entropy    = {builder.entropy():.2f} nats"
      + (f"   KL(p‖q0) = {kl:.2f} nats" if kl is not None else ""))
print()
S = builder.sample_dict(n_samples=100_000)
is_bin = {n: builder.is_binary[builder.var_name_to_idx[n]] for n in S}
for n, col in S.items():
    if is_bin[n]:
        print(f"{n:<28} P(True) = {col.mean():.3f}")
    else:
        q = np.percentile(col, [10, 50, 90])
        print(f"{n:<28} E = {col.mean():9.3f}   "
              f"P10/50/90 = {q[0]:.2f} / {q[1]:.2f} / {q[2]:.2f}")

fig, axes = plot_pairwise(builder.model, builder.params, n_samples=100_000)
plt.show()

In [ ]:
# ---- D) OPTIONAL: uniform vs gaussian on this question ------------------------
# (loss VALUES are constant-shifted between modes — compare readouts, not losses)
for prior in ("uniform", "gaussian"):
    b = DistributionBuilder(variables, ests, prob_penalty="logit",
                            domain_prior=prior, prior_bound_sds=PRIOR_BOUND_SDS)
    b.fit(steps=1500, n_samples=2048, entropy_reg=ENTROPY_REG, seed=0)
    print(f"{prior:<9} P(target) = {b.marginal(TARGET_NAME).probability:.3f}")

# conditional readouts on the samples, e.g.:
# mask = S["some_variable"] > 10
# print(f"P(target | some_variable > 10) = {S[TARGET_NAME][mask].mean():.3f}")